# Validación adversarial: qué columnas delatan el mes

Un clasificador que distingue 201903 de 201910 usando solo las columnas base.
AUC 0,5 significa que los meses son indistinguibles; AUC 1,0, que alguna columna
los separa sola. Las columnas con más peso son las que cambian con el tiempo.

In [8]:
%run 1_home.ipynb

In [9]:
IPC = {
    201901: 189.6101, 201902: 196.7501, 201903: 205.9571, 201904: 213.0517,
    201905: 219.5691, 201906: 225.5370, 201907: 230.4940, 201908: 239.6077,
    201909: 253.7102, 201910: 262.0661, 201911: 273.2158, 201912: 283.4442,
}
MES_REFERENCIA = 201901

def columnas_monetarias(df):
    return [c for c in df.columns
            if c.startswith("m") or c.startswith("Master_m") or c.startswith("Visa_m")]

def deflactar(df, indice, mes_referencia=MES_REFERENCIA, columnas=None):
    out = df.copy()
    columnas = columnas or columnas_monetarias(df)
    factor = df.foto_mes.map(indice).astype("float32")
    out[columnas] = out[columnas].astype("float32").div(factor, axis=0) * indice[mes_referencia]
    return out

1. El clasificador. 'excluir' permite sacar las columnas que ya sabemos que separan solas.

In [10]:
from sklearn.model_selection import train_test_split

def adversarial(df, mes_a=201903, mes_b=201910, semilla=SEMILLAS[0], excluir=()):
    sub = df[df.foto_mes.isin([mes_a, mes_b])]
    cols = [c for c in columnas_base(sub) if c not in set(excluir)]
    X = sub[cols].astype("float32")
    y = (sub.foto_mes == mes_b).astype("int8")

    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=0.3, stratify=y, random_state=semilla)

    m = lgb.train(
        {"objective": "binary", "metric": "auc", "verbosity": -1,
         "seed": semilla, "learning_rate": 0.05, "num_leaves": 31},
        lgb.Dataset(Xtr, label=ytr),
        num_boost_round=300,
        valid_sets=[lgb.Dataset(Xte, label=yte)],
        callbacks=[lgb.early_stopping(30, verbose=False)])

    imp = pd.Series(m.feature_importance("gain"), index=X.columns)
    return m.best_score["valid_0"]["auc"], imp.sort_values(ascending=False)

2. Las delatoras: se saca la que separa sola y se repite, hasta que ninguna domine.

In [11]:
df = cargar_datos()
monetarias = set(columnas_monetarias(df))
delatoras = []

for i in range(12):
    auc, imp = adversarial(df, excluir=delatoras)
    top = imp.index[0]
    print(f"{i:2}  AUC {auc:.4f}   manda {'$' if top in monetarias else ' '} {top}")
    if auc < 0.90:
        print("\nya nadie separa solo")
        break
    delatoras.append(top)

print("\ndelatoras:", delatoras)

 0  AUC 1.0000   manda   tmobile_app
 1  AUC 1.0000   manda   cmobile_app_trx
 2  AUC 1.0000   manda $ mrentabilidad_annual
 3  AUC 1.0000   manda $ mrentabilidad
 4  AUC 1.0000   manda $ mpasivos_margen
 5  AUC 1.0000   manda   ccomisiones_otras
 6  AUC 1.0000   manda $ mcomisiones
 7  AUC 1.0000   manda $ mcomisiones_otras
 8  AUC 1.0000   manda   Visa_fultimo_cierre
 9  AUC 1.0000   manda $ mactivos_margen
10  AUC 0.9998   manda   Master_fultimo_cierre
11  AUC 0.9993   manda   chomebanking_transacciones

delatoras: ['tmobile_app', 'cmobile_app_trx', 'mrentabilidad_annual', 'mrentabilidad', 'mpasivos_margen', 'ccomisiones_otras', 'mcomisiones', 'mcomisiones_otras', 'Visa_fultimo_cierre', 'mactivos_margen', 'Master_fultimo_cierre', 'chomebanking_transacciones']


3. Con las delatoras afuera, el drift se reparte y el ranking sirve.

In [12]:
auc_nom, imp_nom = adversarial(df, excluir=delatoras)

print(f"AUC adversarial sin las delatoras: {auc_nom:.4f}\n")
for c, v in imp_nom.head(25).items():
    print(f"  {'$' if c in monetarias else ' '} {c:42s} {v:14,.0f}")

print(f"\nmonetarias en el top 25: "
      f"{sum(c in monetarias for c in imp_nom.head(25).index)}")

AUC adversarial sin las delatoras: 0.9984

    ctransferencias_recibidas                         975,839
  $ mcomisiones_mantenimiento                         319,771
    Master_Fvencimiento                               191,499
    Visa_Fvencimiento                                 122,811
  $ Master_mfinanciacion_limite                       120,624
  $ Master_mlimitecompra                              105,633
    cpagodeservicios                                   67,837
  $ Visa_mconsumosdolares                              62,621
    Master_fechaalta                                   48,875
  $ Visa_mlimitecompra                                 35,240
    ctarjeta_visa_descuentos                           35,007
  $ Visa_mpagado                                       34,619
  $ mpayroll                                           25,923
    cliente_antiguedad                                 25,682
    Visa_Finiciomora                                   22,450
  $ Visa_mfinanciacion_limi

4. La pregunta que la ganancia no contesta: ¿deflactar borró el drift?

In [14]:
auc_ipc, _ = adversarial(deflactar(df, IPC), excluir=delatoras)

print(f"nominal:     {auc_nom:.4f}")
print(f"deflactado:  {auc_ipc:.4f}")
print(f"diferencia: {auc_nom - auc_ipc:+.4f}") 

nominal:     0.9984
deflactado:  0.9999
diferencia: -0.0015


5. Qué pasa con tmobile_app, la que separaba sola.

In [16]:
print("tipo:", df.tmobile_app.dtype)

df.groupby("foto_mes").tmobile_app.agg(
    nulos=lambda s: float(s.isna().mean()),
    distintos="nunique",
    suma=lambda s: float(s.fillna(0).sum()))

tipo: object


/tmp/ipykernel_5578/1525351157.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  suma=lambda s: float(s.fillna(0).sum()))


,nulos,distintos,suma
foto_mes,,,
201901,1.0,0,0.0
201902,1.0,0,0.0
201903,1.0,0,0.0
201904,1.0,0,0.0
201905,1.0,0,0.0
201906,1.0,0,0.0
201907,0.0,2,inf
201908,0.0,2,inf
201909,0.0,2,inf


1. La inflación no era la fuente principal del drift. Mirá quiénes delatan el mes, en orden:

tmobile_app y cmobile_app_trx — columnas que no existen antes de 201907. La tabla de la celda 5 lo confirma: nulos = 1,0 de enero a junio, 0,0 desde julio. No es inflación, es un corte de registro.
Master_Fvencimiento, Visa_Fvencimiento, Visa_fultimo_cierre, Master_fultimo_cierre — son fechas. Avanzan con el calendario por construcción. Marcan el mes perfectamente y la inflación no tiene nada que ver.
mrentabilidad, mrentabilidad_annual, mpasivos_margen, mactivos_margen — resultados contables del banco, no precios.

Ninguna de esas se arregla deflactando.

2. Deflactar no bajó el drift. Lo subió. 0,9984 → 0,9999. Esto es lo que la ganancia no te podía decir: la corrección ni siquiera hizo su trabajo.

Y hay una explicación mecánica bastante elegante: dividir cada mes por una constante distinta convierte los montos en una huella del mes. Si los valores originales son más o menos redondos, los valores deflactados de marzo caen en una grilla y los de octubre en otra. El árbol puede leer el mes en esa grilla. O sea que deflactar puede agregar una marca temporal en vez de borrarla. Es una hipótesis, pero encaja con el signo.

3. La conclusión principal queda mucho mejor explicada. Se pasa de “deflactar no mejoró, creemos que porque el nivel sirve de marcador temporal” a algo bastante más sólido: la inflación no era el problema dominante en estos datos, y deflactar no removió el drift que sí existía. Por eso no podía mejorar nada.

Un reparo, y es importante. Los dos AUC están contra el techo, 0,9984 y 0,9999. Comparar dos números pegados a 1,0 no dice mucho: podés afirmar que deflactar no bajó el drift, pero no cuantificar nada. El loop cortó en 12 vueltas sin llegar nunca abajo de 0,90.


In [17]:
cols = columnas_monetarias(df)
a = deflactar(df, IPC, mes_referencia=201901)[cols].to_numpy()
b = deflactar(df, IPC, mes_referencia=201906)[cols].to_numpy()

razon = b / a
print("razón mínima y máxima:", np.nanmin(razon), np.nanmax(razon))
print("constante esperada:   ", IPC[201906] / IPC[201901])

razón mínima y máxima: 1.1894777757092052 1.1894777757092057
constante esperada:    1.1894777757092054


/tmp/ipykernel_5578/1397932605.py:5: RuntimeWarning: invalid value encountered in divide
  razon = b / a
